# Scaling-study splits (5 folds per size)

Builds the training sets of the scaling study: pick the videos with `VIDEOS`
(the five folds per size are listed in the next cell), and the notebook writes
annotations for them.

Since the selected videos may carry no polyp annotations, each fragment receives
a bounding-box trajectory sampled from a pool of real annotations and rescaled
to the frame size, with the identity set per video, so the boxes carry no polyp
identity information.


The training dataloader also needs validation and test annotations, which
this notebook does not generate: copy `instances_val.json` and
`instances_test.json` from `data/rc41/annotations` into the output folder
before training on it.

Run **either** PART 4 (a single `instances_train.json`) **or** PART 6
(`N_SPLITS` variants named `instances_train.<i>.json`), not both: the
dataloader picks up every `instances_train*.json` it finds in the folder.


In [ ]:
valtest_videos = {"001-009", "001-010", "001-014", "002-009", "002-010", "002-011", "002-014", "003-009", "003-010", "003-012", "003-013", "003-014", "003-015", "004-009", "004-011", "004-012", "004-013", "004-014", "004-015", }
all_videos = {"001-001","001-002","001-003","001-004","001-005","001-006","001-007","001-008","001-009","001-010","001-011","001-012","001-013","001-014","001-015","002-001","002-002","002-003","002-004","002-005","002-006","002-007","002-008","002-009","002-010","002-011","002-012","002-013","002-014","002-015","003-001","003-002","003-003","003-004","003-005","003-006","003-007","003-008","003-009","003-010","003-011","003-012","003-013","003-014","003-015","004-001","004-002","004-003","004-004","004-005","004-006","004-007","004-008","004-009","004-010","004-011","004-012","004-013","004-014","004-015",}
train_videos = all_videos.difference(valtest_videos)
print(len(valtest_videos))
print(len(all_videos))
print(len(train_videos))

In [ ]:
from collections import Counter

print("videos per cohort in training set =", Counter([v[:3] for v in train_videos]))

In [ ]:
# --- what to build

# Videos to annotate. Use `train_videos` for the full training split, or any
# subset of it (see the next cell for the subsets used in the scaling study).
VIDEOS = list(train_videos)

# Annotations of the videos to annotate: supplies frames, sizes and video ids.
SOURCE_ANNOTATIONS = "data/rc41/annotations/instances_train.json"

# Pool the bounding boxes are sampled from. It must contain real polyp boxes.
POOL_ANNOTATIONS = "data/rc27/annotations/instances_train.json"

# Where the result is written: <OUT_ROOT>/<OUT_NAME>/annotations/...
OUT_ROOT = "data"
OUT_NAME = "rc41_synthbbox"

INCLUDE_HAS_LESION = False  # Disable this if you intend to create a mixed dataset (pos+neg) and use MIL loss
REPLACE = True  # sample annotation with replacement or not - set this to True if the len(annotation_pool) < len(dataset_to_annotate)
SEED = 42
# Number of annotation variants generated in PART 6. The dataloader consumes
# one variant per epoch, so this must match `--max_epochs` of the training run.
N_SPLITS = 25


In [ ]:
# Video subsets of the scaling study: five folds per size.
# Uncomment one pair of lines to rebuild that subset.

# N = 4
# VIDEOS = ['001-008', '002-007', '003-008', '004-005']
# VIDEOS = ['001-003', '002-003', '003-005', '004-008']
# VIDEOS = ['001-012', '002-001', '003-001', '004-005']
# VIDEOS = ['001-001', '002-004', '003-004', '004-008']
# VIDEOS = ['001-003', '002-001', '003-001', '004-007']
# OUT_NAME = "rc41_synthbbox_videos=4_fold=4"

# N = 8
# VIDEOS = ['001-006', '001-008', '002-007', '002-008', '003-006', '003-008', '004-004', '004-005']
# VIDEOS = ['001-003', '001-008', '002-003', '002-013', '003-005', '003-008', '004-005', '004-008']
# VIDEOS = ['001-012', '001-013', '002-001', '002-008', '003-001', '003-004', '004-005', '004-008']
# VIDEOS = ['001-001', '001-008', '002-004', '002-008', '003-004', '003-007', '004-008', '004-010']
# VIDEOS = ['001-003', '001-015', '002-001', '002-008', '003-001', '003-005', '004-007', '004-008']
# OUT_NAME = "rc41_synthbbox_videos=8_fold=4"

# # N = 16
# VIDEOS = ['001-003', '001-006', '001-008', '001-011', '001-012', '002-003', '002-007', '002-008', '002-013', '003-001', '003-003', '003-006', '003-008', '004-001', '004-004', '004-005']
# VIDEOS = ['001-003', '001-004', '001-008', '001-011', '001-013', '002-002', '002-003', '002-008', '002-013', '003-001', '003-005', '003-006', '003-008', '004-005', '004-007', '004-008']
# VIDEOS = ['001-001', '001-002', '001-011', '001-012', '001-013', '002-001', '002-008', '002-012', '002-015', '003-001', '003-004', '003-005', '003-006', '004-001', '004-005', '004-008']
# VIDEOS = ['001-001', '001-005', '001-006', '001-008', '001-012', '002-004', '002-008', '002-013', '002-015', '003-002', '003-004', '003-005', '003-007', '004-006', '004-008', '004-010']
# VIDEOS = ['001-003', '001-004', '001-007', '001-011', '001-015', '002-001', '002-007', '002-008', '002-015', '003-001', '003-002', '003-005', '003-007', '004-005', '004-007', '004-008']
# OUT_NAME = "rc41_synthbbox_videos=16_fold=4"

# # N = 24
# VIDEOS = ['001-003', '001-004', '001-006', '001-008', '001-011', '001-012', '001-015', '002-002', '002-003', '002-005', '002-007', '002-008', '002-013', '003-001', '003-002', '003-003', '003-004', '003-006', '003-008', '004-001', '004-003', '004-004', '004-005', '004-008']
# VIDEOS = ['001-001', '001-003', '001-004', '001-008', '001-011', '001-012', '001-013', '002-001', '002-002', '002-003', '002-004', '002-008', '002-013', '003-001', '003-005', '003-006', '003-007', '003-008', '003-011', '004-002', '004-003', '004-005', '004-007', '004-008']
# VIDEOS = ['001-001', '001-003', '001-004', '001-008', '001-011', '001-012', '001-013', '002-001', '002-002', '002-003', '002-004', '002-008', '002-013', '003-001', '003-005', '003-006', '003-007', '003-008', '003-011', '004-002', '004-003', '004-005', '004-007', '004-008']
# VIDEOS = ['001-001', '001-003', '001-004', '001-008', '001-011', '001-012', '001-013', '002-001', '002-002', '002-003', '002-004', '002-008', '002-013', '003-001', '003-005', '003-006', '003-007', '003-008', '003-011', '004-002', '004-003', '004-005', '004-007', '004-008']
# VIDEOS = ['001-001', '001-003', '001-004', '001-008', '001-011', '001-012', '001-013', '002-001', '002-002', '002-003', '002-004', '002-008', '002-013', '003-001', '003-005', '003-006', '003-007', '003-008', '003-011', '004-002', '004-003', '004-005', '004-007', '004-008']
# OUT_NAME = "rc41_synthbbox_videos=24_fold=4"

# # N = 32
# VIDEOS = ['001-003', '001-004', '001-005', '001-006', '001-007', '001-008', '001-011', '001-012', '001-015', '002-002', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-012', '002-013', '003-001', '003-002', '003-003', '003-004', '003-005', '003-006', '003-008', '004-001', '004-002', '004-003', '004-004', '004-005', '004-008', '004-010']
# VIDEOS = ['001-001', '001-003', '001-004', '001-005', '001-008', '001-011', '001-012', '001-013', '001-015', '002-001', '002-002', '002-003', '002-004', '002-006', '002-007', '002-008', '002-013', '002-015', '003-001', '003-003', '003-005', '003-006', '003-007', '003-008', '003-011', '004-001', '004-002', '004-003', '004-005', '004-007', '004-008', '004-010']
# VIDEOS = ['001-001', '001-002', '001-004', '001-005', '001-006', '001-011', '001-012', '001-013', '001-015', '002-001', '002-002', '002-003', '002-004', '002-006', '002-008', '002-012', '002-013', '002-015', '003-001', '003-002', '003-003', '003-004', '003-005', '003-006', '003-011', '004-001', '004-002', '004-003', '004-004', '004-005', '004-006', '004-008']
# VIDEOS = ['001-001', '001-004', '001-005', '001-006', '001-007', '001-008', '001-011', '001-012', '001-015', '002-002', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-013', '002-015', '003-001', '003-002', '003-004', '003-005', '003-006', '003-007', '003-011', '004-001', '004-002', '004-003', '004-006', '004-007', '004-008', '004-010']
# VIDEOS = ['001-003', '001-004', '001-005', '001-006', '001-007', '001-011', '001-012', '001-013', '001-015', '002-001', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-013', '002-015', '003-001', '003-002', '003-003', '003-005', '003-006', '003-007', '003-008', '004-001', '004-003', '004-005', '004-006', '004-007', '004-008', '004-010']
# OUT_NAME = "rc41_synthbbox_videos=32_fold=4"

# # N = 41
# VIDEOS = ['001-001', '001-002', '001-003', '001-004', '001-005', '001-006', '001-007', '001-008', '001-011', '001-012', '001-013', '001-015', '002-001', '002-002', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-012', '002-013', '002-015', '003-001', '003-002', '003-003', '003-004', '003-005', '003-006', '003-007', '003-008', '003-011', '004-001', '004-002', '004-003', '004-004', '004-005', '004-006', '004-007', '004-008', '004-010']
# VIDEOS = ['001-001', '001-002', '001-003', '001-004', '001-005', '001-006', '001-007', '001-008', '001-011', '001-012', '001-013', '001-015', '002-001', '002-002', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-012', '002-013', '002-015', '003-001', '003-002', '003-003', '003-004', '003-005', '003-006', '003-007', '003-008', '003-011', '004-001', '004-002', '004-003', '004-004', '004-005', '004-006', '004-007', '004-008', '004-010']
# VIDEOS = ['001-001', '001-002', '001-003', '001-004', '001-005', '001-006', '001-007', '001-008', '001-011', '001-012', '001-013', '001-015', '002-001', '002-002', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-012', '002-013', '002-015', '003-001', '003-002', '003-003', '003-004', '003-005', '003-006', '003-007', '003-008', '003-011', '004-001', '004-002', '004-003', '004-004', '004-005', '004-006', '004-007', '004-008', '004-010']
# VIDEOS = ['001-001', '001-002', '001-003', '001-004', '001-005', '001-006', '001-007', '001-008', '001-011', '001-012', '001-013', '001-015', '002-001', '002-002', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-012', '002-013', '002-015', '003-001', '003-002', '003-003', '003-004', '003-005', '003-006', '003-007', '003-008', '003-011', '004-001', '004-002', '004-003', '004-004', '004-005', '004-006', '004-007', '004-008', '004-010']
# VIDEOS = ['001-001', '001-002', '001-003', '001-004', '001-005', '001-006', '001-007', '001-008', '001-011', '001-012', '001-013', '001-015', '002-001', '002-002', '002-003', '002-004', '002-005', '002-006', '002-007', '002-008', '002-012', '002-013', '002-015', '003-001', '003-002', '003-003', '003-004', '003-005', '003-006', '003-007', '003-008', '003-011', '004-001', '004-002', '004-003', '004-004', '004-005', '004-006', '004-007', '004-008', '004-010']
# OUT_NAME = "rc41_synthbbox_videos=41_fold=5"


In [ ]:
len(set(VIDEOS))


In [ ]:
import json
import math
import pathlib

import numpy as np

from polypsense.dataset import InstanceDataset
from polypsense.dataset.subset import SubsetDataset
from polypsense.dataset.fragment import FragmentDataset
from polypsense.dataset.repo.fragmenter import Fragmenter
from polypsense.dataset.repo.identifier import Identifier
from polypsense.dataset.repo.trackleter import Trackleter


def save_coco(coco, path, *, indent=4):
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(coco, f, indent=indent)


def get_coco_stats(coco):
    """
    Extract the following information form a COCO dataset:
    - number of images
    - number of annotations
    - number of sequences
    - number of identites

    If a field is missing, None is retuend.

    Returns:
        A tuple (n_imgs, n_anns, n_seqs, n_ids).
    """
    n_imgs = len(set([img["id"] for img in coco["images"]]))
    n_anns = len(set([ann["id"] for ann in coco["annotations"]]))

    sequence_ids = [img.get("sequence_id") for img in coco["images"]]
    sequence_ids = [sid for sid in sequence_ids if sid is not None]
    n_seqs = len(set(sequence_ids)) if sequence_ids else None

    ann_ids = [ann.get("identity_id") for ann in coco["annotations"]]
    ann_ids = [aid for aid in ann_ids if aid is not None]
    n_ids = len(set(ann_ids)) if ann_ids else None

    return n_imgs, n_anns, n_seqs, n_ids


# PART 1: load source dataset

In [ ]:
rct_path = SOURCE_ANNOTATIONS
pool_path = POOL_ANNOTATIONS


In [ ]:
rct = InstanceDataset.from_instances("fake", rct_path)

In [ ]:
def ds_to_coco(ds):
    return {
        "images": [ds.img_ann(ds.id(i)) for i in range(len(ds))],
        "annotations": [ds.tgt_ann(ds.id(i)) for i in range(len(ds))]
    }

# PART 2: video subset + annotation pool

In [ ]:
def get_video_subset(coco, video_id):
    if not isinstance(video_id, list):
        video_id = [video_id]

    indices = [i for i in range(len(coco)) if coco.img_ann(coco.id(i))["sequence_id"] in video_id]
    vid = SubsetDataset(coco, indices)

    trackleter = Trackleter(
        vid,
        separation_thresh=1,
        iou_thresh=0.1,
        identity_key="identity_id",
        pbar=True,
    )

    fragmenter = Fragmenter(
        trackleter,
        min_tracklet_length=30,
        fragment_length=30,
        stride=1,
        drop_last=True,
        padding_mode=None,
    )

    fragment_ds = FragmentDataset(
        vid,
        fragments=fragmenter.fragments(),
        reduce_img_ann_keys=[
            ("id", "all"),
            ("file_name", "all"),
            ("width", "one"),
            ("height", "one"),
            ("frame_id", "all"),
            ("sequence_id", "one"),
            ("has_lesion", "one")
        ],
        reduce_tgt_ann_keys=[
            ("id", "all"),
            ("image_id", "all"),
            ("category_id", "one"),
            ("bbox", "all"),
            ("area", "all"),
            ("iscrowd", "all"),
            ("identity_id", "one"),
        ],
    )

    return fragment_ds

In [ ]:
video_subset = get_video_subset(rct, VIDEOS)


In [ ]:
len(video_subset)

In [ ]:
def get_annotation_pool(path):
    instance_ds = InstanceDataset.from_instances("fake", path)

    trackleter = Trackleter(instance_ds)
    identifier = Identifier(instance_ds)

    fragmenter = Fragmenter(
        trackleter,
        min_tracklet_length=30,
        fragment_length=30,
        stride=1,
        drop_last=True,
        padding_mode=None,
    )

    fragment_ds = FragmentDataset(
        instance_ds,
        fragments=fragmenter.fragments(),
        reduce_img_ann_keys=[
            ("id", "all"),
            ("file_name", "all"),
            ("width", "one"),
            ("height", "one"),
            ("frame_id", "all"),
            ("sequence_id", "one"),
            ("has_lesion", "one")
        ],
        reduce_tgt_ann_keys=[
            ("id", "all"),
            ("image_id", "all"),
            ("category_id", "one"),
            ("bbox", "all"),
            ("area", "all"),
            ("iscrowd", "all"),
            ("identity_id", "one"),
        ],
    )

    return fragment_ds

In [ ]:
annotation_pool = get_annotation_pool(pool_path)


In [ ]:
len(annotation_pool)

# PART 3: annotate

## 3.1: helpers

In [ ]:
import math

def is_degenerate(bbox, min_size=0):
    x, y, w, h = bbox

    # 1. Check for non-positive dimensions
    if w <= min_size or h <= min_size:
        return True

    # 2. Check for invalid numerical values (NaN or Inf)
    if not all(math.isfinite(val) for val in bbox):
        return True

    return False

In [ ]:
def rescale_bbox(bbox, old_size, new_size, round_to_int=True):
    """
    Rescales a bounding box from an old image size to a new image size.
    
    Args:
        bbox (list/tuple): [x, y, w, h] in the old coordinate space.
        old_size (tuple): (old_width, old_height)
        new_size (tuple): (new_width, new_height)
        round_to_int (bool): If True, returns integers (typical for pixel coords).
        
    Returns:
        list: Rescaled [x, y, w, h]
    """
    old_w, old_h = old_size
    new_w, new_h = new_size

    # Calculate ratios
    width_ratio = new_w / old_w
    height_ratio = new_h / old_h

    # Apply scaling
    x_new = bbox[0] * width_ratio
    y_new = bbox[1] * height_ratio
    w_new = bbox[2] * width_ratio
    h_new = bbox[3] * height_ratio

    if round_to_int:
        return [round(x_new), round(y_new), round(w_new), round(h_new)]
    
    return [x_new, y_new, w_new, h_new]


## 3.2: annotations

In [ ]:
def annotate(ds, pool, *, replace, seed):
    n = len(ds)

    if not replace and n > len(pool):
        raise ValueError(
            "Cannot sample without replacement from an annotation pool that is smaller than dataset to annotate."
        )

    sampled_idx = np.random.default_rng(seed).choice(len(pool), n, replace=replace)

    # img_ann = [ds.img_ann(ds.id(i)) for i in range(n)]
    # tgt_ann = [pool.tgt_ann(pool.id(i)) for i in range(n)]

    # assert len(img_ann) == len(tgt_ann) == n

    images = []
    annotations = []

    annotation_id_seq = 1

    for i in range(n):
        img = ds.img_ann(ds.id(i))
        tgt = pool.tgt_ann(pool.id(sampled_idx[i]))
        old_img = pool.img_ann(pool.id(sampled_idx[i]))

        cur_w, cur_h = img["width"], img["height"]
        old_w, old_h = old_img["width"], old_img["height"]

        for j in range(len(img["id"])):
            images += [
                {
                    "id": img["id"][j],
                    "file_name": img["file_name"][j],
                    "width": img["width"],
                    "height": img["height"],
                    "frame_id": img["frame_id"][j],
                    "sequence_id": img["sequence_id"],
                }
                | ({"has_lesion": img["has_lesion"]} if INCLUDE_HAS_LESION else {})
            ]

            bbox = rescale_bbox(tgt["bbox"][j], (old_w, old_h), (cur_w, cur_h))

            if is_degenerate(bbox):
                raise AssertionError(f"Bbox {i} is degenerate: {bbox}")

            annotations += [
                {
                    "id": annotation_id_seq,  # avoid clashes in case of sampling with replacement
                    "image_id": img["id"][j],
                    "bbox": bbox,
                    "area": tgt["area"][j],
                    "iscrowd": tgt["iscrowd"][j],
                    "category_id": tgt["category_id"],
                    "identity_id": img[
                        "sequence_id"
                    ],  # fake, bboxes are randomly sampled
                }
            ]

            annotation_id_seq += 1

    return {
        "images": images,
        "annotations": annotations,
    }

In [ ]:
vid_coco = annotate(video_subset, annotation_pool, replace=REPLACE, seed=SEED)

In [ ]:
# vid_coco

In [ ]:
get_coco_stats(vid_coco)

# PART 4: save

In [ ]:
save_path = f"{OUT_ROOT}/{OUT_NAME}/annotations/instances_train.json"
print(save_path)


In [ ]:
save_coco(vid_coco, save_path)

# PART 5: check

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def visualize_frames_with_bboxes(
    image_paths,
    bboxes,
    dpi=100,
    figsize=(16, 3)
):
    """
    Visualize a list of 8 images with one bounding box per image.

    Args:
        image_paths (list of str): List of image file paths (len = 8).
        bboxes (list of tuple): List of bounding boxes (len = 8),
            each bbox is (x_min, y_min, width, height).
        dpi (int): DPI of the figure.
        figsize (tuple): Size of the figure.
    """
    assert len(image_paths) == len(bboxes), "Mismatch between images and bboxes"
    n = len(image_paths)

    fig, axes = plt.subplots(1, n, figsize=figsize, dpi=dpi)

    if n == 1:
        axes = [axes]

    for ax, img_path, bbox in zip(axes, image_paths, bboxes):
        # Load image
        img = Image.open(img_path).convert("RGB")

        # Show image
        ax.imshow(img)
        ax.axis("off")

        # Draw bbox
        x, y, w, h = bbox
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=2,
            edgecolor='red',
            facecolor='none'
        )
        ax.add_patch(rect)

    plt.tight_layout()
    plt.show()

In [ ]:
def get_fragments(ds):
    trackleter = Trackleter(ds)
    identifier = Identifier(ds)

    fragmenter = Fragmenter(
        trackleter,
        min_tracklet_length=30,
        fragment_length=30,
        stride=1,
        drop_last=True,
        padding_mode=None,
    )

    fragments = fragmenter.fragments()

    return fragments


In [ ]:
ds = InstanceDataset.from_instances("fake", save_path)
ds_fragments = get_fragments(ds)

In [ ]:
# locate a fragment of a given frame, to eyeball the sampled boxes
PREVIEW_FRAME = "001-001_22500"

preview_idx = None
for i in range(len(ds_fragments)):
    if PREVIEW_FRAME in ds.img_ann(ds_fragments[i][0])["file_name"]:
        preview_idx = i
        print(preview_idx)
        break


In [ ]:
print(len(ds_fragments))
print(ds_fragments[preview_idx])

for id in ds_fragments[preview_idx]:
    print(ds.img_ann(id)["file_name"], ds.tgt_ann(id)["bbox"])

file_names = ["data/rc41/images/" + ds.img_ann(id)["file_name"] for id in ds_fragments[preview_idx]]
bboxes = [ds.tgt_ann(id)["bbox"] for id in ds_fragments[preview_idx]]
visualize_frames_with_bboxes(file_names[::4], bboxes[::4], dpi=72)


# PART 6: generate N splits

In [ ]:
# pad to 4 digits for better sorting
# used below for naming the output file
i_fmt = "{:04d}"

In [ ]:
seeds = np.arange(N_SPLITS) + SEED

for i in range(N_SPLITS):
    save_path = f"{OUT_ROOT}/{OUT_NAME}/annotations/instances_train.{i_fmt.format(i)}.json"
    vid_coco = annotate(video_subset, annotation_pool, replace=REPLACE, seed=seeds[i])
    print(get_coco_stats(vid_coco))
    print(f"Saving {save_path}")
    save_coco(vid_coco, save_path)
